In [ ]:
import re
import csv

expr_file = "GSE221521_gene_expression.xls"
series_file = "GSE221521_series_matrix.txt"
out_file = "GSE221521_gene_expression_labeled.xls"

# -----------------------------------
# 1) Read labels from series_matrix.txt
# -----------------------------------
sample_to_label = {}

with open(series_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("!Sample_title"):
            parts = line.strip().split("\t")[1:]   # skip !Sample_title

            for p in parts:
                p = p.strip().strip('"')

                m = re.search(r'(Control|DM|DR) group (RNA\d+|R_JS\d+)', p)
                if m:
                    label = m.group(1)
                    sample_id = m.group(2)
                    sample_to_label[sample_id] = label
            break

print("Total labeled samples found:", len(sample_to_label))
print("First 10 mappings:", list(sample_to_label.items())[:10])

# -----------------------------------
# 2) Read expression file and relabel header
# -----------------------------------
with open(expr_file, "r", encoding="utf-8") as fin:
    reader = csv.reader(fin, delimiter="\t")
    rows = list(reader)

header = rows[0]
new_header = []
matched = 0

for col in header:
    if col in ["gene_id", "gene_name", "description", "gene_type", "locus"]:
        new_header.append(col)
        continue

    m = re.match(r'^(RNA\d+|R_JS\d+)_(FPKM|count)$', col)
    if m:
        sample_id = m.group(1)
        if sample_id in sample_to_label:
            label = sample_to_label[sample_id]
            new_header.append(f"{label}_{col}")
            matched += 1
        else:
            new_header.append(col)
    else:
        new_header.append(col)

rows[0] = new_header

with open(out_file, "w", encoding="utf-8", newline="") as fout:
    writer = csv.writer(fout, delimiter="\t", lineterminator="\n")
    writer.writerows(rows)

print("Matched expression columns:", matched)
print("Saved:", out_file)
print("First 20 new columns:", new_header[:20])

Total labeled samples found: 193
First 10 mappings: [('RNA1', 'DM'), ('RNA102', 'DR'), ('RNA104', 'DM'), ('RNA108', 'DM'), ('RNA109', 'Control'), ('RNA110', 'DM'), ('RNA114', 'DM'), ('RNA115', 'DM'), ('RNA116', 'DR'), ('RNA117', 'Control')]
Matched expression columns: 386
Saved: GSE221521_gene_expression_labeled.xls
First 20 new columns: ['gene_id', 'gene_name', 'description', 'gene_type', 'locus', 'DM_RNA1_FPKM', 'DR_RNA102_FPKM', 'DM_RNA104_FPKM', 'DM_RNA108_FPKM', 'Control_RNA109_FPKM', 'DM_RNA110_FPKM', 'DM_RNA114_FPKM', 'DM_RNA115_FPKM', 'DR_RNA116_FPKM', 'Control_RNA117_FPKM', 'Control_RNA124_FPKM', 'DM_RNA126_FPKM', 'DM_RNA132_FPKM', 'Control_RNA133_FPKM', 'DR_RNA140_FPKM']


In [ ]:
import pandas as pd

df = pd.read_csv("/content/GSE221521_gene_expression_labeled.xls", sep="\t")
print(df.columns[:30])
print("Total columns:", len(df.columns))

Index(['gene_id', 'gene_name', 'description', 'gene_type', 'locus',
       'DM_RNA1_FPKM', 'DR_RNA102_FPKM', 'DM_RNA104_FPKM', 'DM_RNA108_FPKM',
       'Control_RNA109_FPKM', 'DM_RNA110_FPKM', 'DM_RNA114_FPKM',
       'DM_RNA115_FPKM', 'DR_RNA116_FPKM', 'Control_RNA117_FPKM',
       'Control_RNA124_FPKM', 'DM_RNA126_FPKM', 'DM_RNA132_FPKM',
       'Control_RNA133_FPKM', 'DR_RNA140_FPKM', 'DM_RNA145_FPKM',
       'DR_RNA150_FPKM', 'DM_RNA151_FPKM', 'DM_RNA152_FPKM', 'DR_RNA155_FPKM',
       'Control_RNA160_FPKM', 'DM_RNA165_FPKM', 'Control_RNA167_FPKM',
       'DR_RNA174_FPKM', 'DM_RNA179_FPKM'],
      dtype='object')
Total columns: 391


In [ ]:
fpkm_cols = [col for col in df.columns if col.endswith("_FPKM")]
dr_cols = [col for col in fpkm_cols if col.startswith("DR_")]
control_cols = [col for col in fpkm_cols if col.startswith("Control_")]

print("Total FPKM columns:", len(fpkm_cols))
print("DR columns:", len(dr_cols))
print("Control columns:", len(control_cols))
print("Example DR columns:", dr_cols[:5])
print("Example Control columns:", control_cols[:5])

Total FPKM columns: 193
DR columns: 69
Control columns: 50
Example DR columns: ['DR_RNA102_FPKM', 'DR_RNA116_FPKM', 'DR_RNA140_FPKM', 'DR_RNA150_FPKM', 'DR_RNA155_FPKM']
Example Control columns: ['Control_RNA109_FPKM', 'Control_RNA117_FPKM', 'Control_RNA124_FPKM', 'Control_RNA133_FPKM', 'Control_RNA160_FPKM']


In [ ]:
keep_cols = ["gene_id", "gene_name"] + dr_cols + control_cols
df = df[keep_cols]

print("After keeping DR/Control FPKM:", df.shape)

After keeping DR/Control FPKM: (60675, 121)


In [ ]:
df["gene_name"] = df["gene_name"].astype(str).str.strip()
df = df[df["gene_name"] != ""]
df = df[df["gene_name"].str.lower() != "nan"]

print("After removing empty gene names:", df.shape)

After removing empty gene names: (60675, 121)


In [ ]:
fpkm_cols = dr_cols + control_cols

df["mean_expr"] = df[fpkm_cols].mean(axis=1)
df = df.sort_values("mean_expr", ascending=False)
df = df.drop_duplicates(subset="gene_name", keep="first")
df = df.drop(columns=["mean_expr"])

print("After removing duplicate gene names:", df.shape)

After removing duplicate gene names: (59471, 121)


In [ ]:
min_samples = int(0.2 * len(fpkm_cols))   # at least 20% of samples
df = df[(df[fpkm_cols] > 0).sum(axis=1) >= min_samples]

print("After filtering low-expression genes:", df.shape)

After filtering low-expression genes: (31242, 121)


In [ ]:
import numpy as np

df[fpkm_cols] = np.log2(df[fpkm_cols] + 1)
print("Log transform done")

Log transform done


In [ ]:
import pandas as pd

# Load your file
df = pd.read_csv("GSE221521_DR_Control_FPKM.csv")

print("Original shape:", df.shape)
print(df.columns[:10])

Original shape: (31242, 120)
Index(['gene_name', 'DR', 'DR.1', 'DR.2', 'DR.3', 'DR.4', 'DR.5', 'DR.6',
       'DR.7', 'DR.8'],
      dtype='object')


In [ ]:
sample_cols = [col for col in df.columns if col != "gene_name"]

print("Number of samples:", len(sample_cols))
print(sample_cols[:5])

Number of samples: 119
['DR', 'DR.1', 'DR.2', 'DR.3', 'DR.4']


In [ ]:
df_t = pd.DataFrame(
    df[sample_cols].to_numpy().T,
    index=sample_cols,
    columns=df["gene_name"].values
)

print("Transposed shape:", df_t.shape)

Transposed shape: (119, 31242)


In [ ]:
df_t.to_csv("GSE221521_transposed_DR_Control.csv", index=True)

print("Transposed file saved")

Transposed file saved


In [ ]:
import pandas as pd

df_t = pd.read_csv("GSE221521_transposed_DR_Control.csv")

print("Shape:", df_t.shape)
df_t.head()

Shape: (119, 31243)


,Unnamed: 0,MT-RNR2,S100A9,MT-CO1,MT-CO2,MT-CO3,MT-RNR1,FTL,MT-ND1,MT-ND4,...,PIEZO2,C5orf64,MIR924HG,AL033523.1,LINC02609,MYT1L,MEG8,TRPM3,GABRB3,PEG3
0,DR,12.893180,13.559789,12.771701,12.019843,11.979784,12.032318,12.023987,11.451752,11.631272,...,0.005028,0.000000,0.006275,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.002651
1,DR.1,13.104578,12.559697,13.159648,12.727592,11.843231,11.741509,12.258827,12.074234,12.317882,...,0.005300,0.003726,0.019761,0.000000,0.0,0.002560,0.000000,0.003130,0.0,0.000000
2,DR.2,12.764409,13.541881,13.040206,12.592216,12.726502,11.941926,12.014237,12.106973,12.153423,...,0.000000,0.000000,0.006356,0.002447,0.0,0.005731,0.015347,0.003006,0.0,0.000000
3,DR.3,12.339704,13.587950,12.139918,11.931172,11.703323,11.349380,11.965308,11.352069,11.327643,...,0.010774,0.003793,0.000000,0.000000,0.0,0.000000,0.008154,0.003187,0.0,0.001424
4,DR.4,13.349902,13.952246,12.437290,12.633550,12.378757,12.554389,12.416916,11.718050,11.455981,...,0.000000,0.003665,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.001376


In [ ]:
import pandas as pd

In [ ]:
df1 = pd.read_csv('/content/GSE160306_gene_symbols_updated.csv')
df1

,Sample,TSPAN6,DPM1,SCYL3,FIRRM,CFH,FUCA2,GCLC,NFYA,STPG1,...,LOC105374338,C13orf46,ENSG00000283236,ENSG00000283312,ENSG00000283341,ENSG00000283417,ENSG00000283486,LINC01902,LOC122455340,LOC729732
0,Control,13.296876,14.473577,14.855904,14.081856,12.618348,14.055223,16.924684,15.973886,16.100875,...,14.855904,12.891353,13.787631,11.274930,13.361997,13.073251,12.701134,13.106211,13.251315,14.157047
1,Control,13.055670,14.364616,15.377383,14.339404,13.484874,13.419066,16.518363,16.307698,15.846195,...,14.896788,13.244137,13.836431,11.517020,13.814039,13.134604,12.430274,12.710828,13.406005,13.706553
2,Control,13.445849,14.494166,15.263427,14.229668,12.831553,13.214436,16.729033,16.160798,15.891618,...,14.051851,12.025755,13.744492,11.260701,13.461980,12.745067,13.328754,13.098244,14.074766,13.721254
3,Control,13.812388,14.662366,15.784155,14.532957,14.697344,13.493135,16.100795,16.841618,16.208667,...,13.714660,12.818959,13.764392,11.749387,13.946742,12.452357,12.895812,12.397418,14.353153,12.587136
4,Control,13.475160,14.921105,15.043298,13.933490,13.803471,13.261301,16.958059,16.081452,15.673948,...,13.216314,13.531782,13.769266,11.151620,13.598736,13.479718,12.007213,11.550052,12.910201,13.216314
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,DR,13.567362,14.492131,15.809168,14.327635,13.200165,13.261842,16.232468,16.549285,15.805297,...,14.004710,13.200669,13.720348,11.519279,13.567959,13.478697,12.995041,12.242772,14.092609,13.547808
72,DR,14.806184,14.188908,15.464621,14.185955,15.520754,14.715565,16.928254,16.382395,15.180884,...,14.153708,13.101334,13.499985,11.376729,13.242530,14.805483,11.523102,13.056936,12.711970,14.748321
73,Control,13.455301,14.526534,15.279205,14.307587,13.747046,14.012620,16.670265,16.213922,16.260938,...,15.210361,13.374176,13.597462,11.594921,13.772216,13.126396,12.371974,13.228578,13.589184,13.675295
74,DR,14.156278,14.629324,15.864178,14.480851,12.588662,13.299744,16.512324,16.706663,15.973711,...,13.879815,13.076897,14.125515,11.881480,14.067897,13.387794,12.584424,12.912039,13.333385,13.214132


In [ ]:
df2 = pd.read_csv('/content/GSE221521_transposed_DR_Control.csv')
df2

,Sample,MT-RNR2,S100A9,MT-CO1,MT-CO2,MT-CO3,MT-RNR1,FTL,MT-ND1,MT-ND4,...,PPM1F-AS1,CLTCL1,AL359504.1,CDCA4P1,CCNB1,HMGN5,AC073569.1,KCNE1,AC073130.2,EBAG9P1
0,DR,12.893180,13.559789,12.771701,12.019843,11.979784,12.032318,12.023987,11.451752,11.631272,...,0.473991,0.630272,0.129787,0.786314,0.465728,0.467523,1.303847,1.110256,0.464659,0.636994
1,DR,13.104578,12.559697,13.159648,12.727592,11.843231,11.741509,12.258827,12.074234,12.317882,...,0.405927,0.754600,0.482400,0.327368,0.508868,0.885608,0.000000,0.722601,0.952266,0.370409
2,DR,12.764409,13.541881,13.040206,12.592216,12.726502,11.941926,12.014237,12.106973,12.153423,...,0.465222,0.558760,0.778164,0.315729,0.552917,0.512307,0.156232,0.345733,0.469654,0.507541
3,DR,12.339704,13.587950,12.139918,11.931172,11.703323,11.349380,11.965308,11.352069,11.327643,...,0.451091,0.519352,0.590004,0.931628,0.641241,0.364656,0.313305,0.780423,0.267747,0.376224
4,DR,13.349902,13.952246,12.437290,12.633550,12.378757,12.554389,12.416916,11.718050,11.455981,...,0.476879,0.492362,0.370676,0.000000,0.584187,0.284914,0.000000,0.650916,0.609247,0.365046
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114,Control,14.853635,11.370602,13.602053,13.709240,13.705192,14.268455,12.167742,13.705979,12.947674,...,0.301309,0.401035,0.931662,0.000000,0.348112,0.429993,0.596946,0.400673,0.364309,0.000000
115,Control,12.983228,13.456110,12.642371,12.384391,12.341593,12.179524,12.246978,11.779440,11.848216,...,0.544114,0.557257,0.184239,0.232398,0.811023,0.428832,0.113330,0.936328,0.350403,0.379891
116,Control,13.055578,13.626904,12.623888,12.328568,12.582557,12.071325,12.346621,11.832412,11.834816,...,0.532149,0.542375,0.316233,1.004332,0.482071,0.621076,0.556385,0.992822,0.178766,0.000000
117,Control,12.803411,13.867941,12.273576,12.157953,12.097297,11.938353,12.455578,11.563254,11.419749,...,0.504788,0.889884,0.853819,0.720031,0.515905,0.557568,0.312946,0.271454,0.492899,0.375802


In [ ]:
df1.rename(columns={"Unnamed: 0": "SampleID"}, inplace=True)
df2.rename(columns={"Unnamed: 0": "SampleID"}, inplace=True)

print("SampleID column fixed")

SampleID column fixed


In [ ]:
print("DF1 Shape:", df1.shape)
print("DF2 Shape:", df2.shape)

df1.head()

DF1 Shape: (76, 15067)
DF2 Shape: (119, 16384)


,Sample,TSPAN6,DPM1,SCYL3,FIRRM,CFH,FUCA2,GCLC,NFYA,STPG1,...,LOC105374338,C13orf46,ENSG00000283236,ENSG00000283312,ENSG00000283341,ENSG00000283417,ENSG00000283486,LINC01902,LOC122455340,LOC729732
0,Control,13.296876,14.473577,14.855904,14.081856,12.618348,14.055223,16.924684,15.973886,16.100875,...,14.855904,12.891353,13.787631,11.274930,13.361997,13.073251,12.701134,13.106211,13.251315,14.157047
1,Control,13.055670,14.364616,15.377383,14.339404,13.484874,13.419066,16.518363,16.307698,15.846195,...,14.896788,13.244137,13.836431,11.517020,13.814039,13.134604,12.430274,12.710828,13.406005,13.706553
2,Control,13.445849,14.494166,15.263427,14.229668,12.831553,13.214436,16.729033,16.160798,15.891618,...,14.051851,12.025755,13.744492,11.260701,13.461980,12.745067,13.328754,13.098244,14.074766,13.721254
3,Control,13.812388,14.662366,15.784155,14.532957,14.697344,13.493135,16.100795,16.841618,16.208667,...,13.714660,12.818959,13.764392,11.749387,13.946742,12.452357,12.895812,12.397418,14.353153,12.587136
4,Control,13.475160,14.921105,15.043298,13.933490,13.803471,13.261301,16.958059,16.081452,15.673948,...,13.216314,13.531782,13.769266,11.151620,13.598736,13.479718,12.007213,11.550052,12.910201,13.216314


In [ ]:
df2.head()

,Sample,MT-RNR2,S100A9,MT-CO1,MT-CO2,MT-CO3,MT-RNR1,FTL,MT-ND1,MT-ND4,...,PPM1F-AS1,CLTCL1,AL359504.1,CDCA4P1,CCNB1,HMGN5,AC073569.1,KCNE1,AC073130.2,EBAG9P1
0,DR,12.893180,13.559789,12.771701,12.019843,11.979784,12.032318,12.023987,11.451752,11.631272,...,0.473991,0.630272,0.129787,0.786314,0.465728,0.467523,1.303847,1.110256,0.464659,0.636994
1,DR,13.104578,12.559697,13.159648,12.727592,11.843231,11.741509,12.258827,12.074234,12.317882,...,0.405927,0.754600,0.482400,0.327368,0.508868,0.885608,0.000000,0.722601,0.952266,0.370409
2,DR,12.764409,13.541881,13.040206,12.592216,12.726502,11.941926,12.014237,12.106973,12.153423,...,0.465222,0.558760,0.778164,0.315729,0.552917,0.512307,0.156232,0.345733,0.469654,0.507541
3,DR,12.339704,13.587950,12.139918,11.931172,11.703323,11.349380,11.965308,11.352069,11.327643,...,0.451091,0.519352,0.590004,0.931628,0.641241,0.364656,0.313305,0.780423,0.267747,0.376224
4,DR,13.349902,13.952246,12.437290,12.633550,12.378757,12.554389,12.416916,11.718050,11.455981,...,0.476879,0.492362,0.370676,0.000000,0.584187,0.284914,0.000000,0.650916,0.609247,0.365046


In [ ]:
# keep Sample column as label
df1["Label"] = df1["Sample"]
df2["Label"] = df2["Sample"]

print("Label column created")

Label column created


In [ ]:
# drop old Sample column after copying it to Label
df1 = df1.drop(columns=["Sample"])
df2 = df2.drop(columns=["Sample"])

print("Old Sample column dropped")

Old Sample column dropped


In [ ]:
# clean column names
df1.columns = df1.columns.str.strip()
df2.columns = df2.columns.str.strip()

print("Column names cleaned")

Column names cleaned


In [ ]:
common_genes = list(set(df1.columns).intersection(set(df2.columns)))
common_genes = [g for g in common_genes if g not in ["Label"]]


In [ ]:
# keep only label + common genes
cols = ["Label"] + common_genes

df1_common = df1[cols]
df2_common = df2[cols]

print("Filtered both datasets")
print("DF1 common shape:", df1_common.shape)
print("DF2 common shape:", df2_common.shape)

Filtered both datasets
DF1 common shape: (76, 9433)
DF2 common shape: (119, 9433)


In [ ]:
final_df = pd.concat([df1_common, df2_common], axis=0, ignore_index=True)

print("Final Shape:", final_df.shape)

Final Shape: (195, 9433)


In [ ]:
final_df.to_csv("/content/FINAL_COMBINED_DATASET.csv", index=False)

print("Final dataset saved")

Final dataset saved


In [ ]:
final_df.head()

,Label,COQ9,PABPC1L,TPRG1L,MPPE1,ZNF146,SCOC,ACTR2,TGDS,TMEM192,...,SS18,TRIM62,H2BC4,NKRF,SEMA4C,TMEM107,DYNLL2,TMEM205,SLC43A2,UQCC2
0,Control,16.025628,12.336409,17.309329,15.653138,16.496498,17.390623,18.449478,13.392394,15.606175,...,16.841265,14.354459,14.530531,15.348039,16.674336,15.007002,18.929816,14.949809,17.536280,17.020388
1,Control,15.960102,12.509621,17.279186,15.605717,16.300761,16.851326,18.162973,13.553122,15.830809,...,16.741941,14.373343,14.842613,15.012121,16.988988,15.523052,18.658045,14.879484,17.991895,17.214206
2,Control,15.848842,12.786930,17.306152,15.498367,16.537615,16.837672,18.361164,13.490272,15.745707,...,16.736349,14.431088,15.315686,14.969389,17.607978,15.572541,18.428202,14.688507,17.300138,17.254237
3,Control,15.829655,13.236724,17.108694,15.604973,16.749062,16.986039,18.246535,13.934815,15.896215,...,16.818146,14.355885,14.933209,15.011969,17.088530,15.806350,18.084598,14.651758,17.172735,17.733508
4,Control,15.328748,12.560468,17.275619,15.612704,16.860961,16.912741,18.243922,13.767957,15.808401,...,16.999603,14.779149,15.874373,14.878239,17.011857,16.265691,18.052184,14.767662,18.291914,16.900447


In [ ]:
(final_df == 0).sum().sum()

np.int64(4930)

In [ ]:
total_values = final_df.size
zero_values = (final_df == 0).sum().sum()

percentage = (zero_values / total_values) * 100

print("Zero Percentage:", percentage)

Zero Percentage: 0.26801708133203944
